# Model 2 - Self Attention From Scratch

Ankur 21f2000153

No pretrained weights. Attention written by hand, not using nn.Transformer.

```
question -> embedding -> attention -> mean pool -> 32 numbers
option   -> embedding -> attention -> mean pool -> 32 numbers

concatenate -> 64 numbers -> Linear(64, 1) -> score
```

## Load data

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_FOLDER = "/kaggle/input/competitions/smart-mcq-solver-challenge/"
OPTIONS = ["A", "B", "C", "D", "E"]
LETTER_TO_INDEX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
SEED = 42
VALIDATION_FRACTION = 0.20

np.random.seed(SEED)

train_df = pd.read_csv(DATA_FOLDER + "train.csv")
test_df = pd.read_csv(DATA_FOLDER + "test.csv")

print("train rows:", len(train_df))
print("test rows :", len(test_df))

In [ ]:
PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Identify the correct statement:",
    "Determine the correct option:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]

SUFFIXES = [
    "among the listed options.",
    "from the following choices.",
    "carefully.",
    "based on the given context.",
    "among the list of options.",
]


def clean_prompt(text):
    text = str(text).strip()

    for prefix in PREFIXES:
        if text.startswith(prefix):
            text = text[len(prefix):]
            text = text.strip()
            break

    for suffix in SUFFIXES:
        if text.endswith(suffix):
            text = text[:-len(suffix)]
            text = text.strip()
            break

    return text


train_df["question"] = train_df["prompt"].apply(clean_prompt)
test_df["question"] = test_df["prompt"].apply(clean_prompt)

y_all = train_df["answer"].map(LETTER_TO_INDEX).values

print("before:", train_df["prompt"].iloc[0][:90])
print("after :", train_df["question"].iloc[0][:90])

## Split

In [ ]:
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.strip()


def make_question_key(row):
    option_texts = []
    for letter in OPTIONS:
        option_texts.append(normalize(row[letter]))
    option_texts.sort()
    return "|".join(option_texts)


train_df["key"] = train_df.apply(make_question_key, axis=1)

unique_keys = train_df["key"].unique().tolist()
shuffler = np.random.RandomState(SEED)
shuffler.shuffle(unique_keys)

number_of_validation_keys = int(VALIDATION_FRACTION * len(unique_keys))
validation_keys = set(unique_keys[:number_of_validation_keys])

is_validation = train_df["key"].isin(validation_keys).values

val_df = train_df[is_validation].reset_index(drop=True)
fit_df = train_df[~is_validation].reset_index(drop=True)

y_val = val_df["answer"].map(LETTER_TO_INDEX).values
y_fit = fit_df["answer"].map(LETTER_TO_INDEX).values

shared = set(val_df["key"]) & set(fit_df["key"])

print("total rows       :", len(train_df))
print("unique questions :", train_df["key"].nunique())
print("rows for fitting :", len(fit_df))
print("rows for testing :", len(val_df))
print("shared questions :", len(shared))

## MAP@3

In [ ]:
def map3(scores, labels):
    total = 0.0

    for i in range(len(labels)):
        current_row = scores[i]
        ranking = sorted(range(5), key=lambda j: current_row[j], reverse=True)
        position = ranking.index(labels[i])

        if position < 3:
            total = total + 1 / (position + 1)

    return total / len(labels)


def accuracy(scores, labels):
    best_option = scores.argmax(axis=1)
    return float((best_option == labels).mean())


def macro_f1(scores, labels):
    from sklearn.metrics import f1_score
    best_option = scores.argmax(axis=1)
    return float(f1_score(labels, best_option, average="macro",
                          labels=[0, 1, 2, 3, 4], zero_division=0))


def evaluate(scores, labels, name):
    results = {
        "val_accuracy": accuracy(scores, labels),
        "val_macro_f1": macro_f1(scores, labels),
        "val_map3": float(map3(scores, labels)),
    }
    print(name)
    print("   accuracy :", round(results["val_accuracy"], 4))
    print("   macro F1 :", round(results["val_macro_f1"], 4))
    print("   MAP@3    :", round(results["val_map3"], 4))
    return results

In [ ]:
perfect_scores = np.array([[9, 0, 0, 0, 0], [0, 9, 0, 0, 0]])
second_place = np.array([[0, 9, 0, 0, 0], [9, 0, 0, 0, 0]])
labels = np.array([0, 1])

print("answer ranked first  :", map3(perfect_scores, labels))
print("answer ranked second :", map3(second_place, labels))
print("uniform random guess :", round((1 + 1/2 + 1/3) / 5, 4))

## Settings

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

PAD_ID = 0
UNK_ID = 1

MAX_LENGTH = 32
EMBEDDING_DIM = 32
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001

torch.manual_seed(SEED)

## Tokenizer and vocabulary

Vocabulary is built from the training half only.

In [ ]:
def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-z0-9]+", text)


texts_for_vocabulary = []
for question in fit_df["question"]:
    texts_for_vocabulary.append(question)
for row_number in range(len(fit_df)):
    row = fit_df.iloc[row_number]
    for letter in OPTIONS:
        texts_for_vocabulary.append(row[letter])

word_counts = Counter()
for text in texts_for_vocabulary:
    for word in tokenize(text):
        word_counts[word] += 1

vocabulary = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for word in word_counts:
    vocabulary[word] = len(vocabulary)

print("distinct words:", len(word_counts))
print("vocabulary    :", len(vocabulary))

In [ ]:
def encode_text(text):
    word_ids = []
    for word in tokenize(text):
        if word in vocabulary:
            word_ids.append(vocabulary[word])
        else:
            word_ids.append(UNK_ID)

    word_ids = word_ids[:MAX_LENGTH]
    while len(word_ids) < MAX_LENGTH:
        word_ids.append(PAD_ID)

    return word_ids


def dataframe_to_tensors(dataframe):
    question_ids = []
    for question in dataframe["question"]:
        question_ids.append(encode_text(question))

    option_ids = []
    for row_number in range(len(dataframe)):
        row = dataframe.iloc[row_number]
        this_row = []
        for letter in OPTIONS:
            this_row.append(encode_text(row[letter]))
        option_ids.append(this_row)

    return torch.tensor(question_ids), torch.tensor(option_ids)


questions_fit, options_fit = dataframe_to_tensors(fit_df)
questions_val, options_val = dataframe_to_tensors(val_df)

print("question tensor:", tuple(questions_fit.shape))
print("option tensor  :", tuple(options_fit.shape))

## Self attention

Each word makes a Query, Key and Value. The attention weight from word i to word j is softmax(Query_i . Key_j), and the output is the weighted sum of the Values.

Divided by the square root of the dimension, otherwise the numbers get large and softmax gives almost all the weight to one word.

Padding is set to negative infinity so it gets zero weight.

In [ ]:
class SelfAttention(nn.Module):

    def __init__(self, dimension):
        super().__init__()
        self.make_query = nn.Linear(dimension, dimension)
        self.make_key = nn.Linear(dimension, dimension)
        self.make_value = nn.Linear(dimension, dimension)
        self.dimension = dimension

    def forward(self, x, real_word_mask):
        query = self.make_query(x)
        key = self.make_key(x)
        value = self.make_value(x)

        scores = query @ key.transpose(1, 2)
        scores = scores / math.sqrt(self.dimension)

        padding_mask = ~real_word_mask.unsqueeze(1)
        scores = scores.masked_fill(padding_mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        return weights @ value

## The model

Same encoder is used for the question and the options.

In [ ]:
class ScratchAttentionModel(nn.Module):

    def __init__(self, vocabulary_size, dimension=EMBEDDING_DIM):
        super().__init__()
        self.word_embedding = nn.Embedding(vocabulary_size, dimension,
                                           padding_idx=PAD_ID)
        self.attention = SelfAttention(dimension)
        self.scorer = nn.Linear(dimension * 2, 1)

    def encode_sentence(self, word_ids):
        real_word_mask = (word_ids != PAD_ID)

        x = self.word_embedding(word_ids)
        x = self.attention(x, real_word_mask)

        mask = real_word_mask.unsqueeze(-1).float()
        total = (x * mask).sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1)
        return total / count

    def forward(self, question_ids, option_ids):
        batch_size = option_ids.shape[0]
        number_of_options = option_ids.shape[1]
        sentence_length = option_ids.shape[2]

        question_vector = self.encode_sentence(question_ids)

        flat_options = option_ids.reshape(batch_size * number_of_options,
                                          sentence_length)
        option_vectors = self.encode_sentence(flat_options)

        repeated = question_vector.unsqueeze(1)
        repeated = repeated.expand(batch_size, number_of_options, -1)
        repeated = repeated.reshape(batch_size * number_of_options, -1)

        combined = torch.cat([repeated, option_vectors], dim=1)

        scores = self.scorer(combined)
        return scores.reshape(batch_size, number_of_options)


device = "cuda" if torch.cuda.is_available() else "cpu"
attention_model = ScratchAttentionModel(len(vocabulary)).to(device)

number_of_parameters = sum(p.numel() for p in attention_model.parameters())
print("parameters:", format(number_of_parameters, ","))
print("device    :", device)

## Training

In [ ]:
optimizer = torch.optim.Adam(attention_model.parameters(), lr=LEARNING_RATE)
loss_function = nn.CrossEntropyLoss()

labels_fit = torch.tensor(y_fit)
labels_val = torch.tensor(y_val)

epoch_history = []
best_map3 = 0.0
best_val_scores = None

for epoch in range(1, EPOCHS + 1):

    attention_model.train()
    shuffled_order = torch.randperm(len(questions_fit))
    losses_this_epoch = []

    for start in range(0, len(shuffled_order), BATCH_SIZE):
        batch = shuffled_order[start:start + BATCH_SIZE]

        optimizer.zero_grad()

        predictions = attention_model(questions_fit[batch].to(device),
                                      options_fit[batch].to(device))
        loss = loss_function(predictions, labels_fit[batch].to(device))

        loss.backward()
        optimizer.step()

        losses_this_epoch.append(loss.item())

    attention_model.eval()
    with torch.no_grad():
        val_outputs = attention_model(questions_val.to(device),
                                      options_val.to(device))
        val_loss = loss_function(val_outputs, labels_val.to(device)).item()
        val_scores = val_outputs.cpu().numpy()

    average_loss = float(np.mean(losses_this_epoch))
    this_accuracy = accuracy(val_scores, y_val)
    this_map3 = float(map3(val_scores, y_val))

    if this_map3 > best_map3:
        best_map3 = this_map3
        best_val_scores = val_scores

    epoch_history.append({
        "epoch": epoch,
        "train_loss": average_loss,
        "val_loss": float(val_loss),
        "val_accuracy": this_accuracy,
        "val_map3": this_map3,
    })

    print(f"epoch {epoch:>2}   train_loss {average_loss:.4f}   "
          f"val_loss {val_loss:.4f}   "
          f"accuracy {this_accuracy:.4f}   MAP@3 {this_map3:.4f}")

In [ ]:
attention_results = evaluate(best_val_scores, y_val, "SCRATCH ATTENTION")

## Training curves

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(epoch_history)

figure, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train loss")
axes[0].plot(history["epoch"], history["val_loss"], label="validation loss")
axes[0].set_xlabel("epoch")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["val_map3"])
axes[1].set_xlabel("epoch")
axes[1].set_title("Validation MAP@3")

plt.tight_layout()
plt.show()

Train loss keeps going down but validation loss goes up. The model is memorising the training rows. The best epoch is kept.

## Predictions

In [ ]:
questions_test, options_test = dataframe_to_tensors(test_df)

attention_model.eval()
with torch.no_grad():
    test_scores = attention_model(questions_test.to(device),
                                  options_test.to(device)).cpu().numpy()

top_three = np.argsort(-test_scores, axis=1)[:, :3]

predictions = []
for row in top_three:
    letters = []
    for option_number in row:
        letters.append(OPTIONS[option_number])
    predictions.append(" ".join(letters))

submission = pd.DataFrame({"ID": test_df["id"], "Prediction": predictions})
submission.to_csv("submission_attention.csv", index=False)

print(submission.head())
print("rows:", len(submission))